# HyperParameter Tunning

In [8]:
# Imports
import sys
import os

sys.path.append(os.path.abspath(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\src"))

import pandas as pd
from data_preprocessing import get_feature_table
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from scipy.stats import uniform , randint
from data_preprocessing import processed_dataset , get_feature_table

In [9]:
# Loading preprocessed Dataset

customers = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\dataset\customers.csv")
transactions = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\dataset\transactions.csv")
products = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\SmartCommerce Analytics\dataset\products.csv")

df = processed_dataset(customers , transactions, products)

In [10]:
# 1st Random Try

X = get_feature_table(df)
y = df["lifetime_value_max"]

cat_cols = X.select_dtypes(include=["object" , "string"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

preprocess = ColumnTransformer([("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols) ,
                                ("num" , StandardScaler() , num_cols)])

pipeline = Pipeline([
    ("preprocessing" , preprocess) ,
    ("xgbr" , XGBRegressor(n_jobs = -1 , random_state = 42 , subsample = 0.8 ))
])

param_grid = {
    "xgbr__n_estimators" : randint(300 , 400) ,
    "xgbr__max_depth" :  randint(4,7),
    "xgbr__learning_rate" : uniform(0.01 , 0.09) ,
    "xgbr__gamma" : uniform(0.5 , 0.1) ,
    "xgbr__reg_lambda" : randint(2,7) ,
    "xgbr__reg_alpha" : uniform(0.01 , 0.02) ,
    "xgbr__min_child_weight" : [1,2] 
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param_grid,
    n_iter = 20 ,
    cv = 5
)

srch.fit(X_train , y_train)

model = srch.best_estimator_

print(srch.best_params_)

{'xgbr__gamma': np.float64(0.5801339890558401), 'xgbr__learning_rate': np.float64(0.013094268426319357), 'xgbr__max_depth': 5, 'xgbr__min_child_weight': 1, 'xgbr__n_estimators': 310, 'xgbr__reg_alpha': np.float64(0.011749171809214528), 'xgbr__reg_lambda': 2}


# Considering {max_depth = 5} & {min_child_sample = 2} as fixed baseline

In [13]:
# 2st Random Try

X = get_feature_table(df)
y = df["lifetime_value_max"]

cat_cols = X.select_dtypes(include=["object" , "string"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

preprocess = ColumnTransformer([("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols) ,
                                ("num" , StandardScaler() , num_cols)])

pipeline = Pipeline([
    ("preprocessing" , preprocess) ,
    ("xgbr" , XGBRegressor(n_jobs = -1 , random_state = 42 , subsample = 0.8 , max_depth = 5 , min_child_weight = 2))
])

param_grid = {
    "xgbr__n_estimators" : randint(310 , 400) ,
    "xgbr__learning_rate" : uniform(0.01 , 0.02) ,
    "xgbr__gamma" : uniform(0.45 , 0.15) ,
    "xgbr__reg_lambda" : uniform(0.5 , 3) ,
    "xgbr__reg_alpha" : uniform(0.01 , 0.02) ,
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param_grid,
    n_iter = 20 ,
    cv = 5
)

srch.fit(X_train , y_train)

model = srch.best_estimator_

print(srch.best_params_)

{'xgbr__gamma': np.float64(0.5375122532900355), 'xgbr__learning_rate': np.float64(0.013393057131851155), 'xgbr__n_estimators': 373, 'xgbr__reg_alpha': np.float64(0.013651088553029309), 'xgbr__reg_lambda': np.float64(2.229655939644183)}


# Final Verdict
# ------------------------------------------
# max_depth = 5
# min_child_weight = 2
# gamma = 0.5375
# learning_rate = 0.0133
# alpha = 0.0136
# lambda = 2.229